# GBM Drug Analysis - Data Cleaning and Integration

This notebook handles:
1. Loading GDSC1 and GDSC2 datasets
2. Data cleaning and preprocessing
3. Merging datasets
4. Filtering for GBM cell lines
5. Exploratory data analysis

In [ ]:
# Import libraries
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path('.').absolute().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_processing import GDSCDataLoader
from src import config

%matplotlib inline
sns.set_style('whitegrid')

print(f"Project root: {config.PROJECT_ROOT}")
print(f"Device: {config.DEVICE}")

## 1. Load GDSC Datasets

In [ ]:
# Initialize data loader
loader = GDSCDataLoader()

# Load datasets
print("Loading GDSC datasets...")
gdsc1, gdsc2 = loader.load_gdsc_datasets()

print(f"\nGDSC1 shape: {gdsc1.shape}")
print(f"GDSC2 shape: {gdsc2.shape}")

In [ ]:
# Explore GDSC1
print("GDSC1 columns:")
print(gdsc1.columns.tolist())
print("\nGDSC1 first rows:")
gdsc1.head()

In [ ]:
# Explore GDSC2
print("GDSC2 columns:")
print(gdsc2.columns.tolist())
print("\nGDSC2 first rows:")
gdsc2.head()

## 2. Data Cleaning and Preprocessing

In [ ]:
# Run complete processing pipeline
print("Running data processing pipeline...")
cleaned_data = loader.process_pipeline(filter_gbm=True, save_output=True)

print(f"\nCleaned data shape: {cleaned_data.shape}")
cleaned_data.head()

## 3. Exploratory Data Analysis

In [ ]:
# Summary statistics
print("Data Info:")
cleaned_data.info()

print("\nNumeric columns summary:")
cleaned_data.describe()

In [ ]:
# Distribution of IC50 values
if 'ic50' in cleaned_data.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(cleaned_data['ic50'].dropna(), bins=50, edgecolor='black')
    axes[0].set_xlabel('IC50 (μM)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('IC50 Distribution')
    axes[0].set_xlim(0, 100)  # Focus on main range
    
    # Log scale
    axes[1].hist(np.log10(cleaned_data['ic50'].dropna() + 1), bins=50, edgecolor='black')
    axes[1].set_xlabel('log10(IC50 + 1)')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('IC50 Distribution (Log Scale)')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Number of drugs tested
if 'drug_name' in cleaned_data.columns:
    n_drugs = cleaned_data['drug_name'].nunique()
    print(f"Number of unique drugs: {n_drugs}")
    
    # Top 20 most tested drugs
    top_drugs = cleaned_data['drug_name'].value_counts().head(20)
    
    plt.figure(figsize=(12, 6))
    top_drugs.plot(kind='barh')
    plt.xlabel('Number of Tests')
    plt.title('Top 20 Most Tested Drugs')
    plt.tight_layout()
    plt.show()

In [ ]:
# GBM cell lines distribution
if 'cell_line' in cleaned_data.columns:
    cell_lines = cleaned_data['cell_line'].value_counts()
    print("GBM Cell Lines:")
    print(cell_lines)
    
    plt.figure(figsize=(10, 6))
    cell_lines.plot(kind='bar')
    plt.xlabel('Cell Line')
    plt.ylabel('Number of Drug Tests')
    plt.title('Drug Tests per GBM Cell Line')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Effective drugs (IC50 < 10 μM)
if 'is_effective' in cleaned_data.columns:
    n_effective = cleaned_data['is_effective'].sum()
    total = len(cleaned_data)
    print(f"Effective drug-cell line combinations: {n_effective} / {total} ({n_effective/total*100:.1f}%)")
    
    # Top effective drugs
    effective_drugs = cleaned_data[cleaned_data['is_effective']]
    if len(effective_drugs) > 0:
        top_effective = effective_drugs['drug_name'].value_counts().head(20)
        
        plt.figure(figsize=(12, 6))
        top_effective.plot(kind='barh')
        plt.xlabel('Number of Effective Tests')
        plt.title('Top 20 Most Effective Drugs (IC50 < 10 μM)')
        plt.tight_layout()
        plt.show()

## 4. Save Processed Data

In [ ]:
# Get drug summary statistics
drug_summary = loader.get_drug_summary_statistics(cleaned_data)
print(f"\nDrug summary shape: {drug_summary.shape}")
drug_summary.head(10)

In [ ]:
# Save drug summary
summary_file = config.PROCESSED_DATA_DIR / "drug_summary_stats.csv"
drug_summary.to_csv(summary_file, index=False)
print(f"Drug summary saved to: {summary_file}")

## Summary

This notebook completed the following:
- ✓ Loaded GDSC1 and GDSC2 datasets from RDS files
- ✓ Cleaned and standardized data
- ✓ Filtered for GBM cell lines
- ✓ Handled missing values and outliers
- ✓ Generated exploratory visualizations
- ✓ Saved processed data for downstream analysis

Next: Proceed to `02_feature_engineering.ipynb` for molecular feature extraction.